## Historical Aerial Photograph Index Preparation

This notebook cleans and standardizes the original Minnesota Historical Aerial Photographs Online (MHAPO) point feature class and produces two datasets:
- A cleaned feature class containing the original useful metadata, corrected attributes, and unique photograph identifiers.
- A blank georeferencing index containing the subset of attributes required for the batch georeferencing workflow.

Both datasets share a common unique identifier (source_id) to support joins and subsequent updates.

The original source data are preserved without modification. Processing results and georeferencing status are managed separately from these datasets.

### Planning
- Preserve the original row number for provenance immediately after loading.
- Perform all cleaning and deduplication.
- Sort the final cleaned table.
- Assign permanent source_id values.
- Generate and validate cty_year_id.
- Derive both output tables from that same cleaned GeoDataFrame.

#### Sequence
- 01 Setup and configuration
- 02 Load and inspect source data
- 03 Remove empty fields and standardize field names
- 04 Normalize attribute values
- 05 Spatial county assignment and validation
- 06 Correct known URL errors
- 07 Investigate and resolve duplicate records
- 08 Sort records and generate permanent IDs
- 09 Calculate derived attributes
- 10 Construct the blank georeferencing index
- 11 Validate both output datasets
- 12 Export cleaned feature class and blank index

In [2]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

# Locate the repository root.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\eu01251115\LocalProjects\batch_georef


In [3]:
# Configure file paths
# This is separate from config.yml, to isolate this from batch processing and facilitate manual re-runs

DATA_DIR = PROJECT_ROOT / "data" / "historic_airphoto_index"

SOURCE_PATH = (DATA_DIR / "shapefiles" / "umn_airphoto_index_original.shp")

COUNTIES_PATH = (DATA_DIR / "shapefiles" / "Minnesota_Counties.shp")

CLEANED_PATH = DATA_DIR / "umn_index_cleaned.gpkg"

INDEX_PATH = DATA_DIR / "georef_index_blank.gpkg"

LAYER = "photos"

for path in [SOURCE_PATH, COUNTIES_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

In [4]:
# Load and inspect source data

gdf_original = gpd.read_file(SOURCE_PATH)
counties = gpd.read_file(COUNTIES_PATH)

print("Photo records:", len(gdf_original))
print("Photo index CRS:", gdf_original.crs)

print("County records:", len(counties))
print("County CRS:", counties.crs)

Photo records: 115749
Photo index CRS: EPSG:3857
County records: 87
County CRS: EPSG:3857


In [5]:
# Inspect geometry types and quality.
for name, dataset in [
    ("Photos", gdf_original),
    ("Counties", counties),
]:
    print(f"\n{name}")
    print(dataset.geom_type.value_counts())
    print("Missing geometries:", dataset.geometry.isna().sum())
    print("Empty geometries:", dataset.geometry.is_empty.sum())
    print("Invalid geometries:", (~dataset.geometry.is_valid).sum())


Photos
Point    115749
Name: count, dtype: int64
Missing geometries: 0
Empty geometries: 0
Invalid geometries: 0

Counties
Polygon         85
MultiPolygon     2
Name: count, dtype: int64
Missing geometries: 0
Empty geometries: 0
Invalid geometries: 0


In [6]:
# Create a working copy
gdf = gdf_original.copy().reset_index(drop=True)

# Preserve the original position of each record.
gdf["source_row"] = gdf.index + 1

In [8]:
# Standardize whitespace-only strings as missing values.
text_columns = gdf.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    gdf[col] = gdf[col].replace(
        r"^\s*$", pd.NA, regex=True
    )

In [9]:
# Inspect the schema - profile of the table

field_summary = pd.DataFrame({
    "dtype": gdf.dtypes.astype(str),
    "null_count": gdf.isna().sum(),
    "null_pct": (gdf.isna().mean() * 100).round(2),
    "unique_values": gdf.nunique(dropna=True),
})

display(field_summary)

,dtype,null_count,null_pct,unique_values
flight_cod,str,0,0.00,132
url_link,str,0,0.00,115563
photo_id,str,0,0.00,106857
lat,float64,0,0.00,55847
long_,float64,0,0.00,69028
year_,float64,0,0.00,49
objectid_1,int64,0,0.00,111070
in_fid,int64,0,0.00,111070
near_fid,int64,0,0.00,88
near_dist,float64,0,0.00,1712


In [ ]:
# locate completely empty columns

empty_fields = [
    col
    for col in gdf.columns
    if col != gdf.geometry.name
    and gdf[col].isna().all()
]

print("Entirely empty fields:")
print(empty_fields)

Entirely empty fields:
['metdivfp10', 'decade_dat', 'created_us', 'created_da']


In [11]:
# drop completely empty columns

gdf = gdf.drop(columns=empty_fields)

print("Fields removed:", len(empty_fields))
print("Remaining fields:", len(gdf.columns))

Fields removed: 4
Remaining fields: 34
